In [ ]:
!pip install faiss-cpu rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 57.7 MB/s eta 0:00:00


In [1]:
!pip install faiss-gpu-cu12 rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 MB 53.8 MB/s eta 0:00:00


In [2]:
import abc
from typing import Dict, List, Literal, Optional, Tuple, Union

import numpy as np
import torch

class CrossEncoder(abc.ABC):
    """
    Abstract class for rerankers, providing methods to predict sentence similarity and rank documents based on queries.
    """

    @abc.abstractmethod
    def predict(
            self,
            sentences: Union[
                List[Tuple[str, str]], List[List[str]], Tuple[str, str], List[str]
            ],
            batch_size: Optional[int] = None,
            **kwargs
    ) -> Union[torch.Tensor, np.ndarray]:
        """
        Predicts similarity or relevance scores for pairs of sentences or lists of sentences.

        Args:
            sentences (`Union[List[Tuple[str, str]], List[List[str]], Tuple[str, str], List[str]]`):
                Sentences to predict similarity scores for. Can be a list of sentence pairs, list of sentence lists,
                a single sentence pair, or a list of sentences.
            batch_size (`Optional[int]`, *optional*):
                Batch size for prediction. Defaults to None.

        Returns:
            `Union[torch.Tensor, np.ndarray]`:
                Predicted similarity or relevance scores as a tensor or numpy array.
        """
        raise NotImplementedError


class Reranker(abc.ABC):
    """
    Abstract class for reranking modules that defines methods to rerank search results based on queries.
    """

    @abc.abstractmethod
    def rerank(
            self,
            corpus: Dict[str, Dict[str, str]],
            queries: Dict[str, str],
            results: Dict[str, Dict[str, float]],
            top_k: int,
            batch_size: Optional[int] = None,
            **kwargs
    ) -> Dict[str, Dict[str, float]]:
        """
        Reranks the search results based on the given queries and the initial ranking scores.

        Args:
            corpus (`Dict[str, Dict[str, str]]`):
                A dictionary where keys are document IDs and values are dictionaries containing
                document metadata, such as content or other features.
            queries (`Dict[str, str]`):
                A dictionary where keys are query IDs and values are the corresponding query texts.
            results (`Dict[str, Dict[str, float]]`):
                A dictionary where keys are query IDs and values are dictionaries mapping document
                IDs to their initial relevance scores.
            top_k (`int`):
                The number of top documents to rerank.
            batch_size (`Optional[int]`, *optional*):
                The batch size to use during reranking. Useful for models that process data in
                batches. Defaults to None.
            **kwargs:
                Additional keyword arguments for custom configurations in the reranking process.

        Returns:
            `Dict[str, Dict[str, float]]`:
                The reranked relevance scores, returned as a dictionary mapping query IDs to dictionaries of document IDs and their scores.
        """
        raise NotImplementedError

In [3]:
import logging
from typing import Dict, Optional


logger = logging.getLogger(__name__)


# Adapted from https://github.com/beir-cellar/beir/blob/main/beir/reranking/rerank.py
class CrossEncoderReranker(Reranker):
    """
    A reranker class that utilizes a cross-encoder model from the `sentence-transformers` library
    to rerank search results based on query-document pairs. This class implements a reranking
    mechanism using cross-attention, where each query-document pair is passed through the
    cross-encoder model to compute relevance scores.

    The cross-encoder model expects two inputs (query and document) and directly computes a
    score indicating the relevance of the document to the query. The model follows the
    `CrossEncoder` protocol, ensuring it is compatible with `sentence-transformers` cross-encoder models.

    Methods:
        rerank:
            Takes in a corpus, queries, and initial retrieval results, and reranks
            the top-k documents using the cross-encoder model.
    """

    def __init__(self, model: CrossEncoder):
        """
        Initializes the `CrossEncoderReranker` class with a cross-encoder model.

        Args:
            model (`CrossEncoder`):
                A cross-encoder model implementing the `CrossEncoder` protocol from the `sentence-transformers` library.
        """
        self.model: CrossEncoder = model
        self.results: Dict[str, Dict[str, float]] = {}

    def rerank(
            self,
            corpus: Dict[str, Dict[str, str]],
            queries: Dict[str, str],
            results: Dict[str, Dict[str, float]],
            top_k: int,
            batch_size: Optional[int] = None,
            **kwargs
    ) -> Dict[str, Dict[str, float]]:
        """
        Reranks the top-k documents for each query based on cross-encoder model predictions.

        Args:
            corpus (`Dict[str, Dict[str, str]]`):
                A dictionary representing the corpus, where each key is a document ID and each value is a dictionary
                containing the title and text fields of the document.
            queries (`Dict[str, str]`):
                A dictionary containing query IDs as keys and the corresponding query texts as values.
            results (`Dict[str, Dict[str, float]]`):
                A dictionary containing query IDs and the initial retrieval results. Each query ID is mapped to another
                dictionary where document IDs are keys and initial retrieval scores are values.
            top_k (`int`):
                The number of top documents to rerank for each query.
            batch_size (`Optional[int]`, *optional*):
                The batch size used when passing the query-document pairs through the cross-encoder model.
                Defaults to None.
            **kwargs:
                Additional arguments passed to the cross-encoder model during prediction.

        Returns:
            `Dict[str, Dict[str, float]]`:
                A dictionary containing query IDs as keys and dictionaries of reranked document IDs and their scores as values.
        """
        sentence_pairs, pair_ids = [], []

        for query_id in results:
            if len(results[query_id]) > top_k:
                for doc_id, _ in sorted(
                        results[query_id].items(), key=lambda item: item[1], reverse=True
                )[:top_k]:
                    pair_ids.append([query_id, doc_id])
                    corpus_text = (
                            corpus[doc_id].get("title", "")
                            + " "
                            + corpus[doc_id].get("text", "")
                    ).strip()
                    sentence_pairs.append([queries[query_id], corpus_text])

            else:
                for doc_id in results[query_id]:
                    pair_ids.append([query_id, doc_id])
                    corpus_text = (
                            corpus[doc_id].get("title", "")
                            + " "
                            + corpus[doc_id].get("text", "")
                    ).strip()
                    sentence_pairs.append([queries[query_id], corpus_text])

        #### Starting to Rerank using cross-attention
        logger.info(f"Starting To Rerank Top-{top_k}....")
        rerank_scores = [
            float(score)
            for score in self.model.predict(
                sentences=sentence_pairs, batch_size=batch_size, **kwargs
            )
        ]

        #### Reranker results
        self.results = {query_id: {} for query_id in results}
        for pair, score in zip(pair_ids, rerank_scores):
            query_id, doc_id = pair[0], pair[1]
            self.results[query_id][doc_id] = score

        return self.results

In [4]:
class Lexical(abc.ABC):
    """
    Abstract class for lexical models that defines an interface for calculating relevance scores
    between a query and a set of documents. This abstract class is designed to be implemented by
    classes that calculate document-query relevance using lexical methods such as BM25 or
    other term-based approaches.
    """

    @abc.abstractmethod
    def get_scores(self, query: List[str], **kwargs) -> List[float]:
        """
        Calculates relevance scores for a given query against a set of documents.

        Args:
            query (`List[str]`):
                A tokenized query in the form of a list of words. This represents the query
                to be evaluated for relevance against the documents.

        Returns:
            `List[float]`:
                A list of relevance scores, where each score corresponds to the relevance of
                a document in the indexed corpus to the provided query.
        """
        raise NotImplementedError

class Retrieval(abc.ABC):
    """
    Abstract class for retrieval modules, providing a method to search for the most relevant documents based on queries.
    """

    @abc.abstractmethod
    def retrieve(
            self,
            corpus: Dict[str, Dict[Literal["title", "text"], str]],
            queries: Dict[str, str],
            top_k: Optional[int] = None,
            score_function: Optional[str] = None,
            **kwargs
    ) -> Dict[str, Dict[str, float]]:
        """
        Searches the corpus for the most relevant documents to the given queries.

        Args:
            corpus (`Dict[str, Dict[Literal["title", "text"], str]]`):
                A dictionary where each key is a document ID and each value is another dictionary containing document fields
                (e.g., {'text': str, 'title': str}).
            queries (`Dict[str, str]`):
                A dictionary where each key is a query ID and each value is the query text.
            top_k (`Optional[int]`, *optional*):
                The number of top documents to return for each query. If None, return all documents. Defaults to None.
            score_function (`Optional[str]`, *optional*):
                The scoring function to use when ranking the documents (e.g., 'cosine', 'dot', etc.). Defaults to None.
            **kwargs:
                Additional arguments passed to the search method.

        Returns:
            `Dict[str, Dict[str, float]]`:
                A dictionary where each key is a query ID, and each value is another dictionary mapping document IDs to
                relevance scores (e.g., {'doc1': 0.9, 'doc2': 0.8}).
        """
        raise NotImplementedError

In [5]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
import nltk
import logging
import numpy as np

from typing import Any, Callable, Dict, List, Literal, Optional
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from rank_bm25 import BM25Okapi

logger = logging.getLogger(__name__)

# install nltk resources if not already present
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')


def tokenize_list(input_list: List[str]) -> List[List[str]]:
    """
    Tokenizes a list of strings using the `nltk.word_tokenize` function.

    Args:
        input_list (`List[str]`):
            A list of input strings to be tokenized.

    Returns:
        `List[List[str]]`:
            A list where each element is a list of tokens corresponding to an input string.
    """
    return list(map(word_tokenize, input_list))

class BM25Tokenizer:
    """A custom tokenizer that performs lowercasing, stopword removal, and stemming. A customized tokenizer compared to tokenize_list."""

    def __init__(self):
        """
        Initialize the tokenizer with NLTK's PorterStemmer and English stopwords.
        """
        self.stemmer = PorterStemmer()
        self.stop_words = set(stopwords.words('english'))

    def __call__(self, input_list: List[str]) -> List[List[str]]:
        """Tokenizes, cleans, and stems a list of documents or queries."""
        tokenized_output = []
        for text in input_list:
            # 1. Lowercase and tokenize
            tokens = word_tokenize(text.lower())

            # 2. Filter punctuation, stopwords, and apply stemming
            processed_tokens = [
                self.stemmer.stem(token)
                for token in tokens
                if token.isalnum() and token not in self.stop_words
            ]
            tokenized_output.append(processed_tokens)
        return tokenized_output

class RankBM25Model:
    """
    wrapper class for initializing a 'pretrained' bm25 reranker model using the corpus's tokenized documents.
    ensures that the model adheres to the Lexical protocol.
    """
    def __init__(self, corpus_tokens: List[List[str]]):
        """
        Initializes the BM25 model with the provided tokenized corpus.

        Args:
            corpus_tokens (`List[List[str]]`):
                A list of tokenized documents from the corpus.
        """
        self.bm25 = BM25Okapi(corpus_tokens)

    def get_scores(self, query_tokens: List[str]) -> np.ndarray:
        """
        Computes BM25 scores for the given tokenized query against the corpus.

        Args:
            query_tokens (`List[str]`):
                A list of tokens representing the query.
        """
        return self.bm25.get_scores(query_tokens)

class BM25Retriever(Retrieval):
    """
    A retrieval class that utilizes a lexical model (e.g., BM25) to search for the most relevant documents
    from a given corpus based on the input queries. This retriever tokenizes the queries and uses the provided
    lexical model to compute relevance scores between the queries and documents in the corpus.

    Methods:
        - retrieve: Searches for relevant documents based on the given queries, returning the top-k results.
    """

    def __init__(self, model: Lexical, tokenizer: Callable[[List[str]], List[List[str]]] = tokenize_list):
        """
        Initializes the `BM25Retriever` class with a lexical model and a tokenizer function.

        Args:
            model (`Lexical`):
                A lexical model (e.g., BM25) implementing the `Lexical` protocol, responsible for calculating relevance scores.
            tokenizer (`Callable[[List[str]], List[List[str]]]`, *optional*):
                A function that tokenizes the input queries. Defaults to `tokenize_list`, which uses `nltk.word_tokenize`.
        """
        self.model: Lexical = model
        self.tokenizer: Callable[[List[str]], List[List[str]]] = tokenizer
        self.results: Optional[Dict[str, Any]] = {}

    def retrieve(
            self,
            corpus: Dict[str, Dict[Literal["title", "text"], str]],
            queries: Dict[str, str],
            top_k: Optional[int] = None,
            score_function: Optional[str] = None,
            return_sorted: bool = False,
            **kwargs
    ) -> Dict[str, Dict[str, float]]:
        """
        Searches the corpus for the most relevant documents based on the given queries. The retrieval process involves
        tokenizing the queries, calculating relevance scores using the lexical model, and returning the top-k results
        for each query.

        Args:
            corpus (`Dict[str, Dict[Literal["title", "text"], str]]`):
                A dictionary representing the corpus, where each key is a document ID, and each value is another dictionary
                containing document fields such as 'id', 'title', and 'text'.
            queries (`Dict[str, str]`):
                A dictionary containing query IDs and corresponding query texts.
            top_k (`Optional[int]`, *optional*):
                The number of top documents to return for each query. If not provided, all documents are returned. Defaults to `None`.
            return_sorted (`bool`, *optional*):
                Whether to return the results sorted by score. Defaults to `False`.
            **kwargs:
                Additional keyword arguments passed to the lexical model during scoring.

        Returns:
            `Dict[str, Dict[str, float]]`:
                A dictionary where each key is a query ID, and the value is another dictionary mapping document IDs to relevance scores.
        """
        query_ids = list(queries.keys())
        self.results = {qid: {} for qid in query_ids}

        logger.info("Tokenizing queries with lower cases")
        query_lower_tokens = self.tokenizer([queries[qid].lower() for qid in queries])

        corpus_ids = list(corpus.keys())

        for qid, query in zip(query_ids, query_lower_tokens):
            scores = self.model.get_scores(query)
            top_k_result = np.argsort(scores)[::-1][:top_k]
            for idx in top_k_result:
                self.results[qid][corpus_ids[idx]] = scores[idx]

        return self.results

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [8]:
import json
import numpy as np
import faiss
import pandas as pd
import os
from tqdm import tqdm
from typing import Dict, Literal
from sentence_transformers import SentenceTransformer, CrossEncoder
import gc
import torch

# total vectors: 32225
# nlist clusters: 180 (sqrt(N))

# ========== DENSE RETRIEVER CLASS ==========
class FAISSRetriever:
    """
    Simple FAISS-based retriever for the FinanceRAG competition
    """

    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        """
        Initialize the retriever with an embedding model

        Args:
            model_name: HuggingFace model for embeddings
        """
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        self.dimension = self.model.get_sentence_embedding_dimension()
        print(f"Model embedding dimension: {self.dimension}")

        # FAISS index (will be created when corpus is loaded)
        self.index = None
        self.doc_ids = []  # Maps FAISS index position to document ID

    def load_csv(self, file_path):
        print(f"Loading CSV from {file_path}")
        df = pd.read_csv(file_path)

        assert "_id" in df.columns
        assert "text" in df.columns

        df = df.dropna(subset=["text"])
        df["title"] = df.get("title", "").fillna("")
        df["text"] = df["text"].astype(str)

        records = []
        for _, row in df.iterrows():
            records.append({
                "_id": row["_id"],
                "title": row["title"],
                "text": row["text"]
            })

        print(f"Loaded {len(records)} rows")
        return records


    def build_index(self, corpus_csv, batch_size=64, nlist=16):
        corpus = self.load_csv(corpus_csv)

        documents, self.doc_ids = [], []

        for doc in corpus:
            doc_id = doc["_id"]
            title = doc["title"]
            text = doc["text"]

            # 🔥 IMPROVEMENT: E5-style prompt
            combined = f"passage: {title}. {text}"
            documents.append(combined)
            self.doc_ids.append(doc_id)

        embeddings = self.model.encode(
            documents,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        dim = embeddings.shape[1]

        # 🔥 IMPROVEMENT: IVF + Flat (faster & scalable)
        quantizer = faiss.IndexFlatIP(dim)
        self.index = faiss.IndexIVFFlat(
            quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT
        )

        print("Training FAISS index...")
        self.index.train(embeddings)
        self.index.add(embeddings)

        self.index.nprobe = 32
        print(f"FAISS index built with {self.index.ntotal} vectors")


    def search(self, queries_path, top_k=10, batch_size=32, nprobe=64):
        """
        Search for relevant documents for each query

        Args:
            queries_path: Path to queries JSONL file
            top_k: Number of documents to retrieve per query
            batch_size: Batch size for encoding queries

        Returns:
            Dictionary mapping query_id to {doc_id: score}
        """
        if self.index is None:
            raise ValueError("Index not built. Call build_index() first.")

        # Load queries
        queries_data = self.load_csv(queries_path)

        # Prepare queries
        query_ids = []
        query_texts = []

        print("Preparing queries...")
        for query in queries_data:
            # Adjust these keys based on your JSONL structure
            query_id = query.get('id', query.get('_id', query.get('query_id')))
            query_text = query.get('text', query.get('query', ''))

            query_ids.append(query_id)
            query_texts.append(query_text)

        # Encode queries
        print(f"Encoding {len(query_texts)} queries...")
        query_embeddings = self.model.encode(
            query_texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        # Normalize query embeddings
        faiss.normalize_L2(query_embeddings)

        # Search FAISS index
        print(f"Searching for top {top_k} documents per query...")
        scores, indices = self.index.search(query_embeddings, top_k)

        # Format results
        results = {}
        for i, query_id in enumerate(query_ids):
            results[query_id] = {}
            for j in range(top_k):
                doc_idx = indices[i][j]
                doc_id = self.doc_ids[doc_idx]
                score = float(scores[i][j])
                results[query_id][doc_id] = score

        print(f"Retrieved top {top_k} documents for {len(results)} queries")
        return results

    def search_raw(self, queries_csv, top_k=100, batch_size=64):
        queries = self.load_csv(queries_csv)

        query_ids, query_texts = [], []

        for q in queries:
            query_ids.append(q["_id"])
            query_texts.append(f"query: {q['text']}")

        query_embs = self.model.encode(
            query_texts,
            batch_size=batch_size,
            normalize_embeddings=True
        )

        scores, indices = self.index.search(query_embs, top_k)

        results = {}
        for i, qid in enumerate(query_ids):
            results[qid] = {
                self.doc_ids[idx]: float(scores[i][j])
                for j, idx in enumerate(indices[i])
            }
        return results


    def save_results(self, results, output_path, top_k=10):
        """
        Save results in Kaggle submission format

        Args:
            results: Dictionary from search()
            output_path: Path to save CSV file
            top_k: Number of results to save per query
        """
        print(f"Saving results to {output_path}")

        rows = []
        for query_id, docs in results.items():
            # Sort by score and take top_k
            sorted_docs = sorted(docs.items(), key=lambda x: x[1], reverse=True)[:top_k]

            for doc_id, score in sorted_docs:
                rows.append({'query_id': query_id, 'corpus_id': doc_id})

        df = pd.DataFrame(rows)
        df.to_csv(output_path, index=False)
        print(f"Saved {len(rows)} rows to {output_path}")

# ========== SPARSE RETRIEVER SETUP FUNCTION ==========
def setup_bm25_retriever(corpus_data: Dict[str, Dict[Literal["title", "text"], str]]) -> BM25Retriever:
    """
    Sets up the BM25 model and retriever pipeline.

    Args:
        corpus_data: Your dictionary of all document IDs, titles, and texts.

    Returns:
        An initialized BM25Retriever instance ready for searching.
    """
    # 1. Instantiate the tokenizer
    bm25_tokenizer = BM25Tokenizer()

    # 2. Prepare the corpus for indexing
    # combine title and text for better keyword coverage
    corpus_texts = []
    corpus_ids = []
    for doc_id, doc in corpus_data.items():
        # Combine title and text, ensure lowercase for consistency
        combined_text = f"passage: {doc.get('title', '')} {doc.get('text', '')}".strip()
        corpus_texts.append(combined_text)
        corpus_ids.append(doc_id)

    # 3. Tokenize the entire corpus
    print(f"Tokenizing and stemming {len(corpus_texts)} documents...")
    tokenized_corpus = bm25_tokenizer(corpus_texts)

    # 4. Instantiate the Lexical Model
    print("Building BM25 Index (calculating IDF)...")
    bm25_model = RankBM25Model(tokenized_corpus)

    # 5. Instantiate the final BM25 Retriever
    # Pass the instantiated model and the tokenizer's call method
    retriever = BM25Retriever(
        model=bm25_model,
        tokenizer=bm25_tokenizer
    )

    # (Optional) Store corpus IDs for easier lookup if needed later
    retriever.corpus_ids = corpus_ids

    print("BM25 Retriever setup complete.")
    return retriever

# ========== TASK PROCESSING FUNCTIONS ==========
# SINGLE TASK PROCESSING FUNCTION
def process_single_task(task_name, retriever, top_k=10):
    """
    Process a single task: build index, search, save results

    Args:
        task_name: Name of the task (e.g., 'ConvFinQA')
        retriever: FAISSRetriever instance
        top_k: Number of results per query
    """
    print(f"\n{'='*60}")
    print(f"Processing task: {task_name}")
    print(f"{ '='*60 }\n")

    # Define paths
    corpus_path = f"dataset/{task_name}/corpus.jsonl"
    queries_path = f"dataset/{task_name}/queries.jsonl"
    output_path = f"results/{task_name}_results.csv"

    # Check if files exist
    if not os.path.exists(corpus_path):
        print(f"WARNING: Corpus file not found: {corpus_path}")
        return None
    if not os.path.exists(queries_path):
        print(f"WARNING: Queries file not found: {queries_path}")
        return None

    # Build index from corpus
    retriever.build_index(corpus_path, batch_size=32)

    # Search for queries
    results = retriever.search(queries_path, top_k=top_k, batch_size=32)

    # Save results
    os.makedirs("results", exist_ok=True)
    retriever.save_results(results, output_path, top_k=top_k)

    print(f"\n✓ Task {task_name} completed!")
    return output_path

def expand_query_hyde(query_text: str) -> str:
    """
    Expands the query to improve retrieval.
    In a real competition, you'd use a small LLM here.
    For now, we add domain-specific context to guide the embedding model.
    """
    return f"Financial statement analysis, 10-K report details, and quantitative data for: {query_text}"

# HYBRID TASK PROCESSING FUNCTION
def process_hybrid_task(task_name, faiss_retriever, reranker, top_k_rerank=10, top_k_initial=100):
    """Performs Hybrid Search (FAISS + BM25 + RRF) and Reranking."""
    print(f"\n{'='*60}")
    print(f"Processing HYBRID task: {task_name}")
    print(f"{ '='*60 }\n")

    corpus_path = f"{task_name}_corpus.csv" if task_name in ["FinDER", "FinQABench", "FinanceBench"] else f"{task_name}_corpus_convert.csv"
    queries_path = f"{task_name}_queries.csv"
    output_path = f"{task_name}_hybrid_rerank_results.csv"

    # 1. Setup Data & Build FAISS Index
    corpus_data = faiss_retriever.load_csv(corpus_path) # List of dicts
    faiss_retriever.build_index(corpus_path, batch_size=16) # Builds the index

    # Convert corpus/queries list of dicts to the dict format needed by Reranker/BM25
    corpus_dict = {
        doc.get('id', doc.get('_id')): {'title': doc.get('title', ''), 'text': doc.get('text', '')}
        for doc in corpus_data
    }
    queries_data = faiss_retriever.load_csv(queries_path)
    queries_dict = {}
    expanded_queries_dict = {}
    for query in queries_data:
        qid = query.get('id', query.get('_id'))
        original_text = query.get('text', '')
        queries_dict[qid] = original_text
        # We store the expanded version specifically for retrieval
        expanded_queries_dict[qid] = expand_query_hyde(original_text)

    # 2. Retrieve Candidates (Top N for better Reranker performance)
    print(f"Starting Dense (FAISS) Retrieval: Top {top_k_initial}")
    dense_results = faiss_retriever.search_raw(queries_path, top_k=top_k_initial, batch_size=16)

    print("Starting Sparse (BM25) Retrieval...")
    # Instantiate BM25 Model and Retriever for this task's corpus
    bm25_retriever = setup_bm25_retriever(corpus_dict)
    sparse_results = bm25_retriever.retrieve(corpus=corpus_dict, queries=queries_dict, top_k=top_k_initial)

    print(f"Fusing results using RRF (k_rrf=60)...")
    fused_results = perform_rrf_fusion(dense_results, sparse_results)

    print(f"Reranking fused top-{top_k_rerank} results with Cross-Encoder...")
    final_results = reranker.rerank(
        corpus=corpus_dict,
        queries=queries_dict,
        results=fused_results,
        top_k=top_k_rerank, # Rerank only the desired final amount
        batch_size=8
    )

    os.makedirs("results", exist_ok=True)
    faiss_retriever.save_results(final_results, output_path, top_k=top_k_rerank)

    print(f"\n✓ Hybrid Task {task_name} completed!")
    return output_path

# ========== UTILITY FUNCTIONS ==========
def merge_results(result_files, output_path="submission.csv"):
    """
    Merge all task results into a single submission file

    Args:
        result_files: List of CSV file paths
        output_path: Path for final submission file
    """
    print(f"\n{'='*60}")
    print("Merging results into final submission file")
    print(f"{ '='*60 }\n")

    dfs = []
    for file_path in result_files:
        if file_path and os.path.exists(file_path):
            print(f"Loading {file_path}")
            df = pd.read_csv(file_path)
            dfs.append(df)
            print(f"  → {len(df)} rows")

    if not dfs:
        print("ERROR: No result files to merge!")
        return

    # Concatenate all dataframes
    final_df = pd.concat(dfs, ignore_index=True)

    # Save final submission
    final_df.to_csv(output_path, index=False)

    print(f"\n✓ Final submission saved to: {output_path}")
    print(f"  Total rows: {len(final_df)}")
    print(f"  Unique queries: {final_df['query_id'].nunique()}")
    print(f"\nYou can now submit '{output_path}' to Kaggle!")

# COMBINE RESULTS USING RRF
def perform_rrf_fusion(
    dense_results: Dict[str, Dict[str, float]],
    sparse_results: Dict[str, Dict[str, float]],
    k_rrf: int = 60
) -> Dict[str, Dict[str, float]]:
    """
    Performs Reciprocal Rank Fusion (RRF) to combine dense and sparse results.

    Args:
        dense_results: Results from FAISS (Query ID -> Doc ID -> Score).
        sparse_results: Results from BM25 (Query ID -> Doc ID -> Score).
        k_rrf: RRF constant (default 60 is common).

    Returns:
        A dictionary of fused results (Query ID -> Doc ID -> RRF Score).
    """
    fused_results = {}

    # Iterate over all queries (assuming both dicts have the same query IDs)
    for qid in dense_results.keys():
        all_doc_ids = set(dense_results[qid].keys()) | set(sparse_results[qid].keys())
        rrf_scores = {}

        # 1. Rank Dense Results
        # Sort by score to get the rank (1st result is rank 1)
        ranked_dense = {
            doc_id: rank + 1
            for rank, (doc_id, score) in enumerate(
                sorted(dense_results[qid].items(), key=lambda item: item[1], reverse=True)
            )
        }

        # 2. Rank Sparse Results
        ranked_sparse = {
            doc_id: rank + 1
            for rank, (doc_id, score) in enumerate(
                sorted(sparse_results[qid].items(), key=lambda item: item[1], reverse=True)
            )
        }

        # 3. Apply RRF Formula to all unique documents
        for doc_id in all_doc_ids:
            rank_d = ranked_dense.get(doc_id, 0) # Use 0 if not found
            rank_s = ranked_sparse.get(doc_id, 0) # Use 0 if not found

            # RRF Formula: 1 / (k + rank)
            rrf_score = 0.0
            alpha = 0.7  # Give more weight to dense, or 0.3 if BM25 is performing better on tables
            if rank_d > 0:
                rrf_score += (alpha / (k_rrf + rank_d))
            if rank_s > 0:
                rrf_score += ((1 - alpha) / (k_rrf + rank_s))

            if rrf_score > 0:
                rrf_scores[doc_id] = rrf_score

        # Sort and store the final fused results
        fused_results[qid] = dict(
            sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
        )

    return fused_results

def main():
    """
    Main function to process all tasks
    """
    print("="*60)
    print("FinanceRAG Competition - FAISS Retrieval Pipeline")
    print("="*60)

    # List of all tasks
    tasks = [
        'ConvFinQA',
        'FinDER',
        'FinQA',
        'MultiHiertt',
        'TATQA',
        'FinanceBench',
        'FinQABench'
    ]

    # Initialize retriever with embedding model
    print("\nInitializing retriever and reranker...")
    faiss_retriever = FAISSRetriever(
        #model_name="intfloat/e5-base-v2"
        model_name="intfloat/e5-large-v2"
        #model_name="BAAI/bge-m3"
        # For better results, try:
        # model_name="BAAI/bge-base-en-v1.5"
        # model_name="sentence-transformers/all-mpnet-base-v2"
    )

    print("Initializing reranker...")
    reranker = CrossEncoderReranker(model=CrossEncoder('BAAI/bge-reranker-v2-m3'))
    #reranker = CrossEncoderReranker(model=CrossEncoder('BAAI/bge-reranker-base'))

    # Process each task
    result_files = []
    for task in tqdm(tasks, desc="Processing Tasks"):
        # result_file = process_single_task(task, faiss_retriever, top_k=10)
        result_file = process_hybrid_task(
            task,
            faiss_retriever,
            reranker,
            top_k_rerank=10,
            #top_k_initial=69
            top_k_initial=250
        )
        result_files.append(result_file)

        # --- NEW CODE TO PREVENT OOM ---
        # 1. Clear the FAISS index for the task just finished
        faiss_retriever.index = None

        # 2. Force Python garbage collection
        gc.collect()

        # 3. Empty the CUDA cache so the next task has a "clean" GPU
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # -------------------------------

    # Merge all results
    merge_results(result_files, output_path="submission.csv")

    print("\n" + "="*60)
    print("Pipeline completed successfully!")
    print("="*60)


if __name__ == "__main__":
    main()

FinanceRAG Competition - FAISS Retrieval Pipeline

Initializing retriever and reranker...
Loading embedding model: intfloat/e5-large-v2


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

...


Processing HYBRID task: FinQABench

Loading CSV from FinQABench_corpus.csv
Loaded 92 rows
Loading CSV from FinQABench_corpus.csv
Loaded 92 rows


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Training FAISS index...
FAISS index built with 92 vectors
Loading CSV from FinQABench_queries.csv
Loaded 100 rows
Starting Dense (FAISS) Retrieval: Top 250
Loading CSV from FinQABench_queries.csv
Loaded 100 rows
Starting Sparse (BM25) Retrieval...
Tokenizing and stemming 92 documents...
Building BM25 Index (calculating IDF)...
BM25 Retriever setup complete.
Fusing results using RRF (k_rrf=60)...
Reranking fused top-10 results with Cross-Encoder...
Saving results to FinQABench_hybrid_rerank_results.csv
Saved 1000 rows to FinQABench_hybrid_rerank_results.csv

✓ Hybrid Task FinQABench completed!


Processing Tasks: 100%|██████████| 7/7 [1:57:30<00:00, 1007.16s/it]


Merging results into final submission file

Loading ConvFinQA_hybrid_rerank_results.csv
  → 4210 rows
Loading FinDER_hybrid_rerank_results.csv
  → 2160 rows
Loading FinQA_hybrid_rerank_results.csv
  → 11470 rows
Loading MultiHiertt_hybrid_rerank_results.csv
  → 9740 rows
Loading TATQA_hybrid_rerank_results.csv
  → 16630 rows
Loading FinanceBench_hybrid_rerank_results.csv
  → 1500 rows
Loading FinQABench_hybrid_rerank_results.csv
  → 1000 rows

✓ Final submission saved to: submission.csv
  Total rows: 46710
  Unique queries: 4671

You can now submit 'submission.csv' to Kaggle!

Pipeline completed successfully!


In [ ]:
result_file = [f'{task}_hybrid_rerank_results.csv' for task in ['ConvFinQA', 'FinDER', 'FinQA', 'MultiHiertt', 'TATQA', 'FinanceBench', 'FinQABench']]
print(result_file)
merge_results(result_file, output_path="submission.csv")
print("Submission Generated")

['ConvFinQA_hybrid_rerank_results.csv', 'FinDER_hybrid_rerank_results.csv', 'FinQA_hybrid_rerank_results.csv', 'MultiHiertt_hybrid_rerank_results.csv', 'TATQA_hybrid_rerank_results.csv', 'FinanceBench_hybrid_rerank_results.csv', 'FinQABench_hybrid_rerank_results.csv']

Merging results into final submission file

Loading ConvFinQA_hybrid_rerank_results.csv
  → 4210 rows
Loading FinDER_hybrid_rerank_results.csv
  → 2160 rows
Loading FinQA_hybrid_rerank_results.csv
  → 11470 rows
Loading MultiHiertt_hybrid_rerank_results.csv
  → 9740 rows
Loading TATQA_hybrid_rerank_results.csv
  → 16630 rows
Loading FinanceBench_hybrid_rerank_results.csv
  → 1500 rows
Loading FinQABench_hybrid_rerank_results.csv
  → 1000 rows

✓ Final submission saved to: submission.csv
  Total rows: 46710
  Unique queries: 4671

You can now submit 'submission.csv' to Kaggle!
Submission Generated
